# aiomfac_py Quickstart

Worked examples for **aiomfac_py**, a pure-Python port of [AIOMFAC](https://github.com/andizuend/AIOMFAC) (Aerosol Inorganic–Organic Mixtures Functional groups Activity Coefficients).

Run `bootstrap_colab.ipynb` first (or the setup cell below) so `aiomfac_py` is installed.

1. Basic activity coefficients from an AIOMFAC-web input file
2. Building a mixture by hand with `Component` (a simple aqueous NaCl example)
3. The joint bisulfate + bicarbonate dissociation equilibrium (with Ca2+ precipitation)
4. Going straight from a SMILES string with `aiomfac_py.s2as` (optional, needs `INSTALL_SMILES = True` in the bootstrap notebook)

In [ ]:
# If you have not run bootstrap_colab.ipynb in this session, uncomment and run this setup cell first:
# from pathlib import Path
# import subprocess
# REPO_DIR = Path("/content/aiomfac-python")
# if not REPO_DIR.exists():
#     subprocess.run(["git", "clone", "https://github.com/SeoulTechPSE/aiomfac-python.git", str(REPO_DIR)], check=True)
# get_ipython().run_line_magic("run", f"{REPO_DIR / 'setup_aiomfac.py'} --carbonate")

from pathlib import Path
REPO_DIR = Path("/content/aiomfac-python")

import aiomfac_py as aiomfac
print(f"aiomfac_py {aiomfac.__version__}")

## 1. Activity coefficients from an AIOMFAC-web input file

The package ships with the two example input files from the AIOMFAC repository for quick testing.

In [ ]:
from aiomfac_py import read_input_file, ActivityModel

case = read_input_file(REPO_DIR / "tests/reference/inputs/input_0001.txt")
print(case.title)
for c in case.components:
    print(f"  component {c.number}: {c.name}  subgroups={c.subgroups}")

model = ActivityModel(case.components)
res = model.evaluate(case.fractions[0], case.T_K[0], case.basis)

print()
print("T_K:", res.T_K)
print("mass fractions (wtf):", res.wtf)
print("activity coefficients (neutrals):", res.gamma_neutral)
print("ionic strength (molal):", res.ionic_strength)

## 2. Building a mixture by hand: aqueous NaCl

`Component` takes an AIOMFAC subgroup number and quantity per component. Subgroup 16 is water; ions use AIOMFAC's
own numbering (cations 201–240, anions 241–265) — Na+ is 205, Cl- is 242. See the
[AIOMFAC subgroup list](https://aiomfac.lab.mcgill.ca/subgrouplist.html) for the full table, or use
`aiomfac_py.s2as` (section 4 below) to get subgroups directly from a SMILES string for organics.

In [ ]:
from aiomfac_py import Component, activity_coefficients

water = Component(1, "Water", ((16, 1),))
nacl = Component(2, "NaCl", ((205, 1), (242, 1)))

# 10% NaCl by mass, at 298.15 K:
res = activity_coefficients([water, nacl], [0.9, 0.1], 298.15, basis="mass")
print("gamma (water):", res.gamma_neutral[0])
print("ionic strength (molal):", res.ionic_strength)
print("mean ionic activity-related ln(gamma):", res.ln_gamma)

## 3. Joint bisulfate + bicarbonate dissociation equilibrium

Pass just the neutral SMILES-free ionic subgroups for HSO4-/HCO3- and `ActivityModel` completes the rest of the
system automatically (adding H+, SO4--, CO3--, OH-, CO2(aq) as needed) and solves the coupled 6-unknown
dissociation equilibrium (Fortran `HSO4_and_HCO3_dissociation`) — this is the piece validated to machine
precision against the instrumented Fortran model; see the package README for details.

If Ca2+ (subgroup 221) is also present, the Ca2+ + SO4-- -> CaSO4(s) precipitation pre-step runs automatically
first.

In [ ]:
# Water + NaHSO4 + NaHCO3 (no Ca2+):
water = Component(1, "Water", ((16, 1),))
nahso4 = Component(2, "NaHSO4", ((202, 1), (248, 1)))
nahco3 = Component(3, "NaHCO3", ((202, 1), (250, 1)))

res = activity_coefficients([water, nahso4, nahco3], [0.9, 0.05, 0.05], 298.15, basis="mass")
print("ionic strength (molal):", res.ionic_strength)
print("ln(gamma), all species:", res.ln_gamma)

In [ ]:
# Now with Ca2+ present (CaCl2), triggering the CaSO4(s) precipitation pre-step:
cacl2 = Component(2, "CaCl2", ((221, 1), (242, 2)))
nahso4 = Component(3, "NaHSO4", ((202, 1), (248, 1)))
nahco3 = Component(4, "NaHCO3", ((202, 1), (250, 1)))

res = activity_coefficients([water, cacl2, nahso4, nahco3], [0.85, 0.03, 0.03, 0.03], 298.15, basis="mass")
print("ionic strength (molal):", res.ionic_strength)

## 4. SMILES → AIOMFAC subgroups (`aiomfac_py.s2as`)

Requires the optional `epam.indigo` dependency: re-run the bootstrap notebook with `INSTALL_SMILES = True`, or
run `!pip install epam.indigo` in a cell here. This integrates the
[S2AS tool](https://github.com/andizuend/S2AS__SMILES_to_AIOMFAC) (Amaladhasan et al., 2026) so organic
components can be specified directly by SMILES instead of by hand-picked subgroup counts.

In [ ]:
try:
    from aiomfac_py.s2as import smiles_to_components
except ImportError as e:
    print(e)
    print()
    print("Run: !pip install epam.indigo   (or re-run bootstrap_colab.ipynb with INSTALL_SMILES = True)")
else:
    result = smiles_to_components(["CCCCO"])  # 1-butanol; water is prepended automatically
    for c in result.components:
        print(f"  component {c.number}: {c.name}  subgroups={c.subgroups}")

    model = ActivityModel(result.components)
    res = model.evaluate([0.9, 0.1], 298.15, "mass")  # 90/10 mass-fraction water/butanol
    print()
    print("gamma (water, butanol):", res.gamma_neutral)